In [ ]:
import os
import numpy as np
import xarray as xr
import zarr

import dask
import matplotlib.pyplot as plt

from dask.diagnostics import ProgressBar

In [ ]:
ds = xr.open_zarr('/mnt/tier1/project/p200177/DE_371_bis/meps-2p5km-2020-2025-1h-v2_subdomain_subvars_rechu.zarr', zarr_format=2)
ds = xr.open_zarr('/project/home/p200177/DE_371/datasets/meps-2p5km-2020-2025-1h-v2.zarr', zarr_format=2)


In [ ]:
print(ds)

In [ ]:
ds['maximum'].compute()

In [ ]:
dims = ( "time", "ensemble", "cell")
minn = ds['data'].isel(time=range(31956,31991+1)).mean(dim=dims)
minn.compute()

In [ ]:
dims = ( "time", "ensemble", "cell")
minn = ds['data'].isel(time=range(31991,31991+2)).mean(dim=dims)
minn.compute()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

def plot_meps_variables(ds, start_date, end_date, output_path="variable_trends.png"):
    # 1. Promote 'dates' to a coordinate so we can use .sel() on time
    if 'time' in ds.dims and 'dates' in ds.data_vars:
        ds = ds.assign_coords(time=ds.dates.values)
    
    # 2. Slice the date range
    # We use a slice to ensure dask only pulls relevant chunks
    subset = ds.sel(time=slice(start_date, end_date))
    
    # 3. Aggregate spatial and ensemble dimensions
    # Reducing (time, variable, ensemble, cell) -> (time, variable)
    # We take the mean across all cells and the single ensemble member
    mean_series = subset.data.mean(dim=["cell", "ensemble"]).compute()
    
    # 4. Plotting logic
    var_names = ds.attrs.get("variables", [f"Var {i}" for i in range(28)])
    num_vars = len(var_names)
    
    fig, axes = plt.subplots(nrows=(num_vars + 3) // 4, ncols=4, 
                             figsize=(20, num_vars * 1.2), sharex=True)
    axes = axes.flatten()
    
    for i, var_name in enumerate(var_names):
        ax = axes[i]
        # mean_series shape is (time, variable)
        ax.plot(subset.time, mean_series[:, i], label='Mean')
        ax.set_title(f"Var: {var_name}", fontsize=10)
        ax.grid(alpha=0.3)
        
    # Cleanup empty subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.savefig(output_path)
    print(f"Plot saved to {output_path}")

# Example Usage:
# ds = xr.open_zarr(self.dataset_path)
# plot_meps_variables(ds, "2024-01-01", "2024-01-10")

In [ ]:
plot_meps_variables(ds, "2023-09-30", "2023-10-02")

In [ ]:
import pandas as pd
import numpy as np

def find_temporal_gaps(ds):
    # Extract actual dates (ensure they are loaded as a pandas DatetimeIndex)
    actual_dates = pd.DatetimeIndex(ds.dates.values)
    
    start, end = actual_dates.min(), actual_dates.max()
    
    # Create the ideal hourly range
    ideal_range = pd.date_range(start=start, end=end, freq='h')
    
    # Find the difference
    missing_dates = ideal_range.difference(actual_dates)
    
    print(f"Interval: {start} to {end}")
    print(f"Total expected hours: {len(ideal_range)}")
    print(f"Actual unique hours:   {len(actual_dates.unique())}")
    print(f"Missing hours:         {len(missing_dates)}")
    
    return missing_dates

In [ ]:
find_temporal_gaps(ds)

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

def pinpoint_nan_coordinates(ds):
    # 1. Get metadata for mapping
    timestamps = pd.to_datetime(ds.dates.values)
    var_names = np.array(ds.attrs.get("variables"))
    
    print(f"Scanning for NaNs across {len(var_names)} variables and {len(timestamps)} hours...")

    # 2. Compute a 2D boolean mask: (time, variable)
    # We check if any NaN exists in the spatial 'cell' or 'ensemble' dims
    nan_2d_mask = ds.data.isnull().any(dim=['ensemble', 'cell']).compute()
    
    # 3. Find indices where NaNs exist
    # time_indices and var_indices will be arrays of integers
    time_indices, var_indices = np.where(nan_2d_mask)
    
    if len(time_indices) == 0:
        print("Success: No NaNs found in the dataset.")
        return pd.DataFrame()

    # 4. Map indices to actual values and create a summary
    nan_reports = []
    for t_idx, v_idx in zip(time_indices, var_indices):
        nan_reports.append({
            "timestamp": timestamps[t_idx],
            "variable": var_names[v_idx],
            "time_index": t_idx,
            "variable_index": v_idx
        })
    
    df_nans = pd.DataFrame(nan_reports)
    
    # 5. Professional Summary
    print(f"\nCRITICAL: Found {len(df_nans)} variable-hour intersections with NaNs.")
    print("\nSummary of NaNs per variable:")
    print(df_nans['variable'].value_counts())
    
    return df_nans

# Usage:
# nan_registry = pinpoint_nan_coordinates(ds)

In [ ]:
nan_registry = pinpoint_nan_coordinates(ds)

In [ ]:
nan_registry.to_csv("nans.csv")

In [ ]:
nan_registry['time_index'].unique()

In [ ]:
df = pd.read_csv("nans.csv")

In [ ]:
print(df["timestamp"].unique())

In [ ]:
df['time_index'].unique()